## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# Smoothed Historical CTR Baseline

## 1. Objective

Rank each impression's candidates by smoothed historical click-through rate. Unlike click-count popularity, this score accounts for the number of exposures. It is the standalone reference for the historical-CTR signal used by the MLP cold-user fallback. The model does not use embeddings or fit neural weights.

## 2. Scoring rule

For article $a$ in an impression on day $d$,

$$s(a,d)=\frac{C_{a,<d}+\alpha\mu_{train}}{E_{a,<d}+\alpha}.$$

$C$ counts clicked exposures, $E$ counts all exposures, $\alpha=20$ is the prior strength and $\mu_{train}$ is clicks divided by exposures over the training period. An unseen article receives the prior rather than an unsupported zero or one. With increasing exposure, the observed CTR receives more weight.

## 3. Temporal protocol

Use the fixed five-day training prior. Validation uses exposures from the five training days. With validation updates enabled, test uses all six preceding days, matching the MLP feature setting. No scored day's labels contribute to that day's counts. The prior is a training-fitted statistic, not an event-by-event prior. Frozen click-count and lagged-CTR results therefore have different information windows.


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features


In [2]:
import json
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from utils.metrics import ndcg_at_k

CTR_ALPHA = 20.0
STATS_USE_VAL_FOR_TEST = True
NDCG_KS = (5, 10)
RECALL_KS = (5, 10)
SPARSE_MAX = 5
METRIC_NAMES = ["AUC", "MRR", "nDCG@5", "nDCG@10", "Recall@5", "Recall@10"]
OUT_DIR = PATHS["outputs"] / "base_models" / "popularity_smoothed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reject an inconsistent setting when prepared MLP metadata is available.
metadata_path = PATHS["features"] / "feature_meta.json"
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text())
    assert CTR_ALPHA == metadata["leakage"]["ctr_alpha"]
    assert STATS_USE_VAL_FOR_TEST == metadata["leakage"]["stats_use_val_for_test"]


## 4. Candidate exposure tables

Retain every validation/test candidate and its original position. Build counts before any negative sampling; otherwise the exposure denominator would be biased. Candidate labels are used only for historical counts or held-out evaluation, according to their day.


In [3]:
beh, _ = load_preprocessed(PATHS)
cand = beh[["impression_id", "day", "split", "impressions"]].copy()
cand["candidate"] = cand.pop("impressions").str.split()
cand = cand.explode("candidate", ignore_index=True)
cand[["news_id", "label"]] = cand.pop("candidate").str.rsplit("-", n=1, expand=True)
cand["label"] = cand["label"].astype(np.int8)
cand["slot"] = cand.groupby("impression_id").cumcount()
train = cand[cand["split"] == "train"]
CTR_PRIOR = float(train["label"].mean())
assert np.isfinite(CTR_PRIOR)
print(f"Training CTR prior: {CTR_PRIOR:.6f}; alpha: {CTR_ALPHA}")


Training CTR prior: 0.041012; alpha: 20.0


## 5. Evaluation definitions

Use the existing MLP impression evaluator so this baseline can be compared with its fallback: MRR averages reciprocal ranks over clicked candidates, AUC uses average ranks for ties, and nDCG uses binary relevance. Stable descending ordering preserves candidate order for tied ranking scores. User cohorts are history length 0, 1–4 and at least 5.

The cold-article slice uses zero clicks in the same historical window as the score. Impressions without a clicked cold article are excluded from that slice; clicked warm articles are removed while non-clicked candidates remain. Empty cohorts are reported as undefined. Overall evaluation keeps full slates with at least one click. Recall is retained as an additional source diagnostic.


In [4]:
def impression_metrics(labels, scores):
    order = np.argsort(-scores, kind="stable")
    y = labels[order]
    n_pos = y.sum()
    ranks = np.arange(1, len(y) + 1)
    disc = 1.0 / np.log2(ranks + 1)
    out = {"MRR": float((y / ranks).sum() / n_pos)}
    for k in NDCG_KS:
        ideal = disc[: min(int(n_pos), k)].sum()
        out[f"nDCG@{k}"] = ndcg_at_k(labels, y, k)
    for k in RECALL_KS:
        out[f"Recall@{k}"] = float(y[:k].sum() / n_pos)
    n_neg = len(y) - n_pos
    if n_neg > 0:
        r = rankdata(scores)
        out["AUC"] = float(
            (r[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
        )
    else:
        out["AUC"] = np.nan
    return out


def evaluate(Ad, scores):
    """Returns (metrics table per slice, per-impression metrics)."""
    rows, cold_rows = [], []
    for g in range(len(Ad["n"])):
        s, k = Ad["start"][g], Ad["n"][g]
        lab, sc = Ad["label"][s : s + k], scores[s : s + k]
        rows.append(impression_metrics(lab, sc))
        cold = Ad["is_cold_article"][s : s + k] == 1
        cold_pos = (lab == 1) & cold
        if cold_pos.any():
            keep = ~((lab == 1) & ~cold)
            m = impression_metrics(cold_pos[keep].astype(np.float32), sc[keep])
            m["g"] = g
            cold_rows.append(m)
    per_imp = pd.DataFrame(rows)
    hl = Ad["hist_len"]
    slices = {
        "all": np.ones(len(hl), bool),
        "cold_user": hl == 0,
        "sparse_user": (hl > 0) & (hl < SPARSE_MAX),
        "warm_user": hl >= SPARSE_MAX,
    }
    table = []
    for name, m in slices.items():
        sub = per_imp[m]
        table.append(
            {
                "slice": name,
                "n_impressions": int(m.sum()),
                "n_users": int(pd.unique(Ad["user_id"][m]).size),
                **{k: sub[k].mean() for k in METRIC_NAMES},
            }
        )
    cold_df = pd.DataFrame(cold_rows)
    n_cold_art_users = (
        int(pd.unique(Ad["user_id"][cold_df["g"].values]).size) if len(cold_df) else 0
    )
    table.append(
        {
            "slice": "cold_article",
            "n_impressions": len(cold_df),
            "n_users": n_cold_art_users,
            **{
                k: (cold_df[k].mean() if len(cold_df) else np.nan) for k in METRIC_NAMES
            },
        }
    )
    return pd.DataFrame(table), per_imp


## 6. Validation and test ranking

Evaluate both smoothed CTR and frozen training click-count scores under the same MLP evaluator. Compare cold-user performance to assess whether an exposure-adjusted prior is useful when no personal history exists. This comparison does not establish calibrated CTR for a live homepage or causal gains from slate rotation.


In [5]:
train_counts = train.groupby("news_id")["label"].sum()
metrics = []
for split in ["val", "test"]:
    score_day = int(beh.loc[beh["split"] == split, "day"].min())
    cutoff = score_day if STATS_USE_VAL_FOR_TEST else min(score_day, 5)
    history = cand[cand["day"] < cutoff]
    counts = history.groupby("news_id")["label"].agg(clicks="sum", exposures="size")
    frame = cand[cand["split"] == split].copy()
    clicks = frame["news_id"].map(counts["clicks"]).fillna(0).to_numpy()
    exposures = frame["news_id"].map(counts["exposures"]).fillna(0).to_numpy()
    frame["score"] = (
        (clicks + CTR_ALPHA * CTR_PRIOR) / (exposures + CTR_ALPHA)
    ).astype(np.float32)
    frame["is_cold_article"] = (clicks == 0).astype(np.int8)
    frame["click_count_score"] = (
        frame["news_id"].map(train_counts).fillna(0).astype(np.float32)
    )
    groups = frame.groupby("impression_id", sort=False).size()
    impressions = beh.set_index("impression_id").loc[groups.index]
    lengths = impressions["history"].str.split().str.len().to_numpy()
    counts_per_impression = groups.to_numpy()
    data = {
        "start": np.r_[0, np.cumsum(counts_per_impression)[:-1]],
        "n": counts_per_impression,
        "label": frame["label"].to_numpy(np.float32),
        "hist_len": lengths,
        "user_id": impressions["user_id"].to_numpy(),
        "is_cold_article": frame["is_cold_article"].to_numpy(),
    }
    # Source MLP evaluation omits zero-click impressions.
    assert (frame.groupby("impression_id")["label"].sum() > 0).all()
    for name, column in [
        ("smoothed_ctr", "score"),
        ("click_count", "click_count_score"),
    ]:
        table, _ = evaluate(data, frame[column].to_numpy())
        table.insert(0, "model", name)
        table.insert(0, "split", split)
        metrics.append(table)
    save_table(frame, OUT_DIR / f"{split}_scores")
metrics = pd.concat(metrics, ignore_index=True)
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
(OUT_DIR / "run_config.json").write_text(
    json.dumps(
        {
            "alpha": CTR_ALPHA,
            "prior": CTR_PRIOR,
            "validation_updates_for_test": STATS_USE_VAL_FOR_TEST,
            "evaluator": "MLP source definitions",
        },
        indent=2,
    )
)
display(metrics)


,split,model,slice,n_impressions,n_users,AUC,MRR,nDCG@5,nDCG@10,Recall@5,Recall@10
0,val,smoothed_ctr,all,30270,20179,0.573909,0.235642,0.241991,0.303998,0.355772,0.534857
1,val,smoothed_ctr,cold_user,772,476,0.574780,0.236279,0.243532,0.304987,0.359244,0.537577
2,val,smoothed_ctr,sparse_user,3247,2657,0.565588,0.263665,0.271994,0.331408,0.403909,0.578244
3,val,smoothed_ctr,warm_user,26251,17046,0.574912,0.232157,0.238234,0.300579,0.349716,0.529411
4,val,smoothed_ctr,cold_article,22543,16161,0.501450,0.183633,0.170659,0.230262,0.269151,0.446382
5,val,click_count,all,30270,20179,0.516296,0.215018,0.216807,0.277841,0.325255,0.502948
6,val,click_count,cold_user,772,476,0.539234,0.226045,0.235757,0.295423,0.362111,0.534546
7,val,click_count,sparse_user,3247,2657,0.525472,0.246697,0.254404,0.315681,0.387707,0.569772
8,val,click_count,warm_user,26251,17046,0.514486,0.210775,0.211599,0.272644,0.316446,0.493754
9,val,click_count,cold_article,22543,16161,0.376256,0.161143,0.150494,0.203067,0.244689,0.402498


## 7. Interpretation and limitations

Compare AUC, MRR and nDCG@5/10 within the same split and cohort. A positive difference from click counts can reflect exposure normalization, the training prior or the additional past-day observations. It should not be attributed solely to smoothing. Run a frozen-window sensitivity experiment to isolate the temporal-update effect. Training-based standardization of hist_ctr in the MLP is monotonic and preserves its ranking apart from possible floating-point ties. This notebook reports the raw smoothed score.

Neither popularity policy personalizes scores among users arriving on the same day. The fallback combines this signal with a trained model only for empty-history users; the standalone baseline applies it to every user.
